In [1]:
!pip install sentence-transformers transformers torch scikit-learn

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import torch

In [3]:
document = """
Cybersecurity is the practice of protecting systems and networks from digital attacks.
Phishing is a common attack where users are tricked into giving personal information.

Malware includes viruses, worms, and ransomware.
Ransomware encrypts files and demands payment to restore access.

Intrusion Detection Systems monitor network traffic to detect suspicious activities.
Machine learning is increasingly used to improve cybersecurity defenses.
"""

In [4]:
def chunk_text(text, chunk_size=30):

    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks

chunks = chunk_text(document)

print("Number of chunks:", len(chunks))
print(chunks[0])

Number of chunks: 2
Cybersecurity is the practice of protecting systems and networks from digital attacks. Phishing is a common attack where users are tricked into giving personal information. Malware includes viruses, worms, and


In [5]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedder.encode(chunks)

print(chunk_embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(2, 384)


In [6]:
query = "What is phishing?"

In [7]:
query_embedding = embedder.encode([query])

In [8]:
similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

In [9]:
top_k = 3

top_indices = np.argsort(similarities)[-top_k:][::-1]

relevant_chunks = [chunks[i] for i in top_indices]

for chunk in relevant_chunks:
    print(chunk)

Cybersecurity is the practice of protecting systems and networks from digital attacks. Phishing is a common attack where users are tricked into giving personal information. Malware includes viruses, worms, and
ransomware. Ransomware encrypts files and demands payment to restore access. Intrusion Detection Systems monitor network traffic to detect suspicious activities. Machine learning is increasingly used to improve cybersecurity defenses.


In [10]:
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

model = AutoModelForCausalLM.from_pretrained("distilgpt2")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [11]:
context = " ".join(relevant_chunks)

prompt = f"""
Context: {context}

Question: {query}

Answer:
"""

In [12]:
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    inputs["input_ids"],
    max_length=150,
    do_sample=True,
    temperature=0.7
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Context: Cybersecurity is the practice of protecting systems and networks from digital attacks. Phishing is a common attack where users are tricked into giving personal information. Malware includes viruses, worms, and ransomware. Ransomware encrypts files and demands payment to restore access. Intrusion Detection Systems monitor network traffic to detect suspicious activities. Machine learning is increasingly used to improve cybersecurity defenses.

Question: What is phishing?

Answer:
In many cases, phishing is an ongoing attempt to steal and damage public infrastructure. The Internet and cybersecurity professionals who conduct the research and the research are the victims of many attacks. The risk of phishing is extremely high. The most common attack occurs when a computer system or network is compromised
